<a href="https://colab.research.google.com/github/eliza-aurora-carling/Admin/blob/main/Eliza_CFG_Assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CFG Assignment 2 - Eliza Carling

# Hello! This project will help relieve any boredom you have.
# I will find out what weather you like, and what energy level you have.
# I will suggest an activity for you to try with the number of participants it requires.
# The BoredAPI should work fine unless there are connectivity issues... it has both not worked and worked for me in practice.
# BONUS: i will tell you a random interesting fact :)


import requests
import json
import random
import datetime

def activity_suggestion():
    try:
        url = "https://www.boredapi.com/api/activity"
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        data = response.json()
        return data.get("activity", "Read a book"), data.get("participants", 1)
    except requests.exceptions.RequestException as e:
        print(f"Error while getting activity from API: {e}. Falling back to local suggestions.")
        activities = [
            ("Go for a run", 1),
            ("Read a book", 1),
            ("try a new recipe", 1),
            ("Watch a film", 1),
            ("Play a solo board game e.g. sudoku", 1),
            ("Do some gardening outside", 1),
            ("Call a friend or family member", 1),
            ("Learn a new language for an hour", 1),
            ("Do a puzzle", 1),
            ("Go for a bike ride outside", 1),
            ("Have a picnic", 2),
            ("Visit a museum virtually", 1),
            ("Meditate or yoga session", 1),
            ("Write a letter", 1),
            ("Do some DIY at home", 1),
            ("Listen to a podcast", 1),
            ("draw or paint something", 1),
            ("Dance to some jazz music", 1),
            ("Clean your room while listening to jazz music", 1),
            ("Do some stretches", 1)
        ]
        activity, participants = random.choice(activities)
        return activity, participants

def fact_suggestion():
    try:
        url = "https://uselessfacts.jsph.pl/random.json?language=en"
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        data = response.json()
        return data["text"]
    except requests.exceptions.RequestException as e:
        print(f"Error getting the fact from API: {e}. Falling back to default fact.")
        return "Bees can recognise human faces."


def check_match_compatibility(energy, participants, weather, user_weather):
    score = 0
    reasons = []

    if energy >= participants * 3:
        score += 1
        reasons.append(f"Your energy ({energy}) is high enough for {participants} person activity")
    elif energy >= participants * 2:
        score += 1
        reasons.append(f"Your energy ({energy}) is medium, so you can do the activity but you should rest after")
    else:
        reasons.append(f"You have low energy ({energy}) for {participants} person activity - maybe choose to rest instead")


 # using string slicing on weather word
    weather_slice = user_weather[:3].lower()
    if weather_slice in weather.lower():
        score += 1
        reasons.append(f"luckily, weather '{user_weather}' matches your preference")
    else:
        reasons.append(f"Weather is {weather} - not  ideal but still fine")

    return score, reasons

def save_results(activity, fact, energy, participants, weather, user_weather, score, reasons, verdict):
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"activity_suggester_{timestamp}.txt"

    with open(filename, "w") as f:
        f.write("=" * 50 + "\n")
        f.write("ACTIVITY SUGGESTER REPORT\n")
        f.write("=" * 50 + "\n\n")

        f.write(f"Your energy level: {energy}/10\n")
        f.write(f"Your weather preference: {user_weather}\n")
        f.write(f"Current weather: {weather}\n\n")

        f.write(f"Suggested activity: {activity}\n")
        f.write(f"Participants needed: {participants}\n\n")

        f.write(f"Fun fact: {fact}\n\n")

        f.write(f"Match score: {score}/2\n")
        f.write("Reasons:\n")
        for r in reasons:
            f.write(f"  - {r}\n")
        f.write(f"\nFinal verdict: {verdict}\n")

    return filename

def main():
    print("   Easter activity suggester")
    print("   Find what to do based on your energy levels!!")


    # Get user input
    energy_input = input("Rate your energy 1-10 (1=exhausted, 10=super energetic): ")

    # Boolean!!
    if energy_input.isdigit():
        energy = int(energy_input)
        if energy < 1:
            energy = 1
        elif energy > 10:
            energy = 10
    else:
        print("Using energy level 5 as default")
        energy = 5

    weather_pref = input("What kind of weather do you prefer? (sunny/rainy/cloudy): ").strip().lower()

    # list of possible weathers (using a tuple!)
    weather_options = ("sunny", "rainy", "cloudy", "windy", "snowy")
    current_weather = random.choice(weather_options[:3])

    print(f"\n Checking current weather... It's {current_weather} today.")
    print(" Finding an activity for you \n")

    activity, participants = activity_suggestion()
    fact = fact_suggestion()

    print("-" * 40)
    print(f"Activity idea: {activity[:60]}...")
    print(f"Did you know? {fact[:70]}...")
    print("-" * 40 + "\n")

    # check compatibility match
    match_score, reason_list = check_match_compatibility(energy, participants, current_weather, weather_pref)

    # Boolean decision
    if match_score >= 2:
        verdict = "Yes, it's a great match for your energy and weather."
    elif match_score == 1:
        verdict = "Maybe, could be fun."
    else:
        verdict = "Consider resting today. Low match, maybe try again tomorrow."

    # display results using a loop
    print("results:\n")
    for reason in reason_list:
        print(f"   • {reason}")

    print(f"\n Match score: {match_score}/2")
    print(f"\n{verdict}\n")

    # save to file
    saved_file = save_results(activity, fact, energy, participants, current_weather, weather_pref, match_score, reason_list, verdict)

    print(f" Results saved to: {saved_file}")
    print("\nThanks for using my activity suggester! Happy Easter!\n")

if __name__ == "__main__":
    main()

   Easter activity suggester
   Find what to do based on your energy levels!!
